# A — Colab에서 Gemma 4 E4B 품질 비교

사진·영상 입력 → 해상도/토큰/프레임 조건 변경 → 원문·측정값 저장 → HTML 리포트.

기본값은 **모의 실행**입니다. 모델 성능 결과를 만들지 않습니다. 실제 CUDA 추론은 `MOCK=False`로 바꾼 뒤 실행합니다.

Based-On: none — 대응하는 LMS 멀티모달 벤치마크 예제가 없어 작성한 project_extension입니다.

## 1. 코드·데이터·패키지 준비

Colab에 `filename-model-quality-bench` 코드 폴더와 `filename_eval24_1080` 데이터 폴더를 준비하세요. 제공 ZIP을 각각 풀면 됩니다. Google Drive를 쓰면 다음 셀의 두 경로를 Drive 경로로 바꿉니다.

Colab 메뉴에서 GPU 런타임을 선택합니다. 실제 실행 전 아래 설치 명령을 별도 셀에서 실행하세요. 이 노트북의 기본 모의 실행 검증은 설치·모델 다운로드를 수행하지 않습니다.

```python
%pip install -r /content/filename-model-quality-bench/requirements-colab.txt
```

모델 접근에 로그인이 필요하면 Hugging Face 약관 동의 후 `huggingface_hub.notebook_login()`으로 로그인하세요. 토큰을 코드나 출력에 적지 마세요.

공개 저장소의 코드는 Colab 셀에서 다음 명령으로 준비할 수 있습니다. 평가 데이터는 별도로 준비합니다.

```python
!git clone https://github.com/yooongZa/filename-model-quality-bench.git /content/filename-model-quality-bench
```

In [ ]:
from pathlib import Path
import sys

# Colab에서는 /content 경로. 로컬에서는 이 노트북 폴더 또는 예제 루트에서 실행.
if Path('/content').exists():
    PROJECT = Path('/content/filename-model-quality-bench')
else:
    PROJECT = Path.cwd()
    if PROJECT.name == 'notebooks':
        PROJECT = PROJECT.parent
DATASET = PROJECT / 'data' / 'filename_eval24_1080'

MOCK = True
SUITE = 'smoke'  # baseline / resolution / vision / output / frames / selection
CASE_IDS = ['I01', 'V01']  # 전체 24개는 None
MAX_NEW_REQUESTS = None

assert (PROJECT / 'a_quality.py').exists(), f'코드 폴더를 확인하세요: {PROJECT}'
assert (DATASET / 'model_inputs.jsonl').exists(), f'데이터 폴더를 확인하세요: {DATASET}'
sys.path.insert(0, str(PROJECT))
print('코드:', PROJECT, '\n데이터:', DATASET, '\n모의 실행:', MOCK)

## 2. 실행 조건 확인

설정 원본을 수정하지 않고 이번 노트북에서 사용할 복사본을 만듭니다. 원문·JSON 스키마·입력 해시를 결과에 기록합니다. 모델 응답에서 제목을 생성하지 않으며 태그·근거·지명·검토 필요 여부의 4개 필드를 검사합니다.

In [ ]:
from filename_bench.common import load_config
from filename_bench.runner import run

config = load_config(PROJECT / 'configs/a_quality.json')
config['dataset_root'] = str(DATASET)
config['run_dir'] = str(PROJECT / 'runs/notebook_a')
config['case_ids'] = CASE_IDS
# 처음 확보한 backend_*.json의 resolved_revision을 아래에 고정할 수 있습니다.
# config['backend']['revision'] = '확인한 Hugging Face commit SHA'
plan = run(config, kind='a', suite=SUITE, mock=MOCK, dry_run=True)
print('측정 예정 요청 수:', plan['planned_requests'])

## 3. 실행

`MOCK=False`이면 이 셀에서 모델을 다운로드·로드하고 실제 GPU 추론을 시작합니다. 우선 smoke로 사진과 3·12프레임을 확인하세요. 이후 SUITE를 바꿔 한 변수씩 비교합니다.

`resume=True`는 같은 실험의 완료한 요청을 건너뜁니다. 입력·설정·코드가 바뀌었다면 새 run_dir를 사용해야 합니다. Colab 연결이 종료될 수 있으므로 결과 폴더를 보존 가능한 위치로 지정하세요.

In [ ]:
result = run(config, kind='a', suite=SUITE, mock=MOCK,
             resume=True, limit=MAX_NEW_REQUESTS)
print(result)

## 4. 결과 확인과 사람 평가

형식 검사는 자동으로 집계됩니다. 관찰 근거·동의어·환각·행동 변화는 review.csv에서 확인합니다. 참조 답안은 독립적인 사람 검수를 마치지 않았습니다. 모의 응답의 형식 통과를 모델의 품질 성과로 해석하지 마세요.

In [ ]:
import csv
from IPython.display import display, HTML, FileLink

run_dir = Path(result['run_dir'])
with (run_dir / 'summary.csv').open(encoding='utf-8-sig', newline='') as f:
    summary = list(csv.DictReader(f))
display(summary)
display(HTML((run_dir / 'report.html').read_text(encoding='utf-8')))
print('결과 폴더:', run_dir)
print('사람 평가표:', run_dir / 'review.csv')

## 5. 다음 실험

1. resolution·vision·output·frames·selection을 각각 비교합니다. baseline의 다른 값은 유지합니다.
2. 유망한 설정 2개를 전체 24개에서 재검증합니다. 별도 촬영 묶음으로 최종 결과를 확인합니다.
3. 선택한 설정을 B의 baseline으로 옮겨 Windows 8GB VRAM에서 다시 측정합니다. BF16 Colab 결과와 양자화 GGUF의 품질 차이도 실제 출력으로 확인하세요.

실제 A100 실행·로컬 GPU 실행은 이 파일을 만든 환경에서 검증하지 않았습니다. 자세한 실행·평가 기준은 코드 폴더의 README.md에 있습니다.